# vLLM server on Google Colab

This notebook serves `Qwen/Qwen2.5-14B-Instruct-AWQ` with vLLM and exposes its OpenAI-compatible API through ngrok.

Before running it:
1. Select **Runtime > Change runtime type > T4 GPU** (or a better GPU).
2. In Colab's **Secrets** panel, add `NGROK_AUTHTOKEN` and `VLLM_API_KEY`.
3. Optionally add `HF_TOKEN` to reduce Hugging Face download rate limits.
4. Do not share the ngrok URL or API key. Stop the runtime when testing is complete.

In [3]:
import warnings
warnings.filterwarnings('ignore')

!nvidia-smi
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
!pip install vllm pyngrok

Wed Sep  2 05:38:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             27W /   70W |    9807MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
import subprocess
import time

import requests
from google.colab import userdata
from pyngrok import ngrok

MODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"
PORT = 8000
VLLM_API_KEY = userdata.get("VLLM_API_KEY")

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    pass

command = [
    "vllm", "serve", MODEL,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--api-key", VLLM_API_KEY,
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",
]

log_file = open("/tmp/vllm.log", "w")
vllm_process = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT)
print(f"Started vLLM with process ID {vllm_process.pid}. Model loading can take several minutes.")

Started vLLM with process ID 6490. Model loading can take several minutes.


In [5]:
health_url = f"http://127.0.0.1:{PORT}/health"
deadline = time.time() + 600

while time.time() < deadline:
    if vllm_process.poll() is not None:
        log_file.flush()
        raise RuntimeError("vLLM stopped during startup. Inspect /tmp/vllm.log.")
    try:
        if requests.get(health_url, timeout=5).ok:
            print("vLLM is ready.")
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError("vLLM did not become ready within 10 minutes.")

vLLM is ready.


In [6]:
!cat /tmp/vllm.log

(APIServer pid=6490) INFO 09-02 05:42:21 [api_utils.py:333] 
(APIServer pid=6490) INFO 09-02 05:42:21 [api_utils.py:333]        █     █     █▄   ▄█
(APIServer pid=6490) INFO 09-02 05:42:21 [api_utils.py:333]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.28.0
(APIServer pid=6490) INFO 09-02 05:42:21 [api_utils.py:333]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-14B-Instruct-AWQ
(APIServer pid=6490) INFO 09-02 05:42:21 [api_utils.py:333]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=6490) INFO 09-02 05:42:21 [api_utils.py:333] 
(APIServer pid=6490) INFO 09-02 05:42:21 [api_utils.py:272] non-default args: {'model_tag': 'Qwen/Qwen2.5-14B-Instruct-AWQ', 'enable_auto_tool_choice': True, 'tool_call_parser': 'hermes', 'host': '0.0.0.0', 'api_key': ['xhS064jhqywKeTTV64A66q6VHUQ-b5JKY8U7TSMCNtg'], 'model': 'Qwen/Qwen2.5-14B-Instruct-AWQ', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85}
(APIServer pid=6490) Traceback (most recent call last):
(APIServer pid=6490)   File "/usr/local/bin/vllm", line

In [7]:
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
tunnel = ngrok.connect(PORT, bind_tls=True)
public_url = tunnel.public_url

print("Add these values to task1/.env, then restart FastAPI:")
print("LLM_BACKEND=vllm")
print(f"VLLM_BASE_URL={public_url}/v1")
print(f"VLLM_MODEL={MODEL}")
print("VLLM_API_KEY=<the same value stored in your Colab VLLM_API_KEY secret>")

Add these values to task1/.env, then restart FastAPI:
LLM_BACKEND=vllm
VLLM_BASE_URL=https://7272-34-125-223-147.ngrok-free.app/v1
VLLM_MODEL=Qwen/Qwen2.5-14B-Instruct-AWQ
VLLM_API_KEY=<the same value stored in your Colab VLLM_API_KEY secret>


In [8]:
headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Reply with: vLLM is working"}],
    "temperature": 0,
}
response = requests.post(
    f"{public_url}/v1/chat/completions",
    headers=headers,
    json=payload,
    timeout=120,
)
response.raise_for_status()
print(response.json()["choices"][0]["message"]["content"])

vLLM is working


## Stop the public server

Run the following cell when testing is finished. Colab and ngrok URLs are temporary, so a new runtime requires updating `VLLM_BASE_URL` again.

In [9]:
ngrok.disconnect(public_url)
vllm_process.terminate()
vllm_process.wait(timeout=30)
log_file.close()
print("vLLM and the ngrok tunnel have stopped.")

vLLM and the ngrok tunnel have stopped.
